<a href="https://www.kaggle.com/code/erlangs/mesin-belajar?scriptVersionId=321374592" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ==============================================================================
# 1. IMPORT LIBRARIES & CONFIGURATION
# ==============================================================================
import os
import re
import string
import warnings
import random
from collections import Counter
import numpy as np
import pandas as pd

# ML & Evaluation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Deep Learning (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Global Config
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EMOTION_LABELS = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

print(f'🚀 Running on: {DEVICE}')

🚀 Running on: cpu


In [2]:
# ==============================================================================
# 2. DATA LOADING & LIGHT PREPROCESSING
# ==============================================================================
MELD_RAW = '/kaggle/input/datasets/erlangs/datameld/MELD.Raw'
csv_paths = {
    'train': os.path.join(MELD_RAW, 'train', 'train_sent_emo.csv'),
    'dev':   os.path.join(MELD_RAW, 'dev_sent_emo.csv'),
    'test':  os.path.join(MELD_RAW, 'test_sent_emo.csv'),
}

# Fallback check
if not os.path.isfile(csv_paths['train']):
    alt = os.path.join(MELD_RAW, 'train_sent_emo.csv')
    if os.path.isfile(alt): csv_paths['train'] = alt

datasets = {}
for split, path in csv_paths.items():
    if os.path.isfile(path):
        datasets[split] = pd.read_csv(path, usecols=['Utterance', 'Emotion'])
    else:
        raise FileNotFoundError(f"Path tidak ditemukan: {path}")

df_train = datasets['train']
df_dev   = datasets['dev']
df_test  = datasets['test']

def clean_text_for_chat(text):
    """Pembersihan ramah emosi: STOP WORDS & TANDA BACA PENTING TETAP DIPERTAHANKAN"""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    # Hanya pertahankan huruf, spasi, tanda seru, dan tanda tanya
    text = re.sub(r'[^a-z!?\s]', '', text)
    # Beri spasi pada tanda baca agar terpisah sebagai token tersendiri
    text = re.sub(r'(!+)', ' ! ', text)
    text = re.sub(r'(\?+)', ' ? ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
for df in [df_train, df_dev, df_test]:
    df['clean_text'] = df['Utterance'].apply(clean_text_for_chat)

# Hapus jika benar-benar kosong total
df_train = df_train[df_train['clean_text'].str.strip().str.len() > 0].reset_index(drop=True)
df_dev   = df_dev[df_dev['clean_text'].str.strip().str.len() > 0].reset_index(drop=True)
df_test  = df_test[df_test['clean_text'].str.strip().str.len() > 0].reset_index(drop=True)

# Encode Labels
le = LabelEncoder()
le.fit(EMOTION_LABELS)
y_train = le.transform(df_train['Emotion'])
y_dev   = le.transform(df_dev['Emotion'])
y_test  = le.transform(df_test['Emotion'])

print(f"Dataset Size -> Train: {len(df_train)}, Dev: {len(df_dev)}, Test: {len(df_test)}")

Dataset Size -> Train: 9987, Dev: 1108, Test: 2610


In [3]:
# ==============================================================================
# 3. CLASSICAL ML BASELINE (Multinomial NB)
# ==============================================================================
print("\n--- Training Baseline Model (Naive Bayes) ---")
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(df_train['clean_text'])
X_test_tfidf  = tfidf.transform(df_test['clean_text'])

nb_model = MultinomialNB(alpha=0.5)
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
print(f"Naive Bayes Test Accuracy: {accuracy_score(y_test, nb_preds):.4f}")


--- Training Baseline Model (Naive Bayes) ---
Naive Bayes Test Accuracy: 0.5165


In [5]:
# ==============================================================================
# 4. VOCABULARY & PYTORCH DATASET BUILDER
# ==============================================================================
MAX_VOCAB = 10000
MAX_LEN = 32 # Diperkecil dari 64 karena sebaran data kalimat pendek dominan < 20 kata

word_counts = Counter()
for text in df_train['clean_text']:
    word_counts.update(text.split())

# Reservasi id 0 untuk PAD dan id 1 untuk UNK
most_common = word_counts.most_common(MAX_VOCAB - 2)
vocab = {'<PAD>': 0, '<UNK>': 1}
for w, _ in most_common:
    vocab[w] = len(vocab)

class TokenizedDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=32):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        tokens = self.texts[idx].split()[:self.max_len]
        # Peta kata ke ID, gunakan 1 (<UNK>) jika tidak terdaftar
        ids = [self.vocab.get(t, 1) for t in tokens]
        
        # Padding manual
        pad_len = self.max_len - len(ids)
        ids = ids + [0] * pad_len
        
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

BATCH_SIZE = 64
train_loader = DataLoader(TokenizedDataset(df_train['clean_text'].tolist(), y_train, vocab, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
dev_loader   = DataLoader(TokenizedDataset(df_dev['clean_text'].tolist(), y_dev, vocab, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TokenizedDataset(df_test['clean_text'].tolist(), y_test, vocab, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

In [6]:
# ==============================================================================
# 5. COMPACT BiLSTM + ATTENTION MODEL Arsitektur Baru
# ==============================================================================
class CompactBiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=64, num_classes=7, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # 1 Layer saja agar tidak overfit di teks pendek bercakapan
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, 
                           bidirectional=True, batch_first=True)
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # Buat masker penanda token PAD (B, L, 1) agar attention tidak fokus pada isi kosong
        mask = (x != 0).float().unsqueeze(-1)
        
        emb = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(emb) # Output format: (B, L, 2*H)
        
        # Mekanisme Scoring Atensi
        attn_scores = self.attention(lstm_out)
        # Menenggelamkan nilai bobot token PAD menjadi minus tak terhingga sebelum softmax
        attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_w = torch.softmax(attn_scores, dim=1)
        
        # Konteks vektor kombinasi linear bobot atensi dengan representasi LSTM
        context = (lstm_out * attn_w).sum(dim=1)
        return self.fc(self.dropout(context))

model = CompactBiLSTMAttention(len(vocab), num_classes=len(EMOTION_LABELS)).to(DEVICE)

# Penyeimbang bobot kelas data imbalanced
class_counts = np.bincount(y_train, minlength=7).astype(float)
class_weights = torch.tensor(1.0 / (class_counts + 1e-6), dtype=torch.float32).to(DEVICE)
class_weights = class_weights / class_weights.sum() * 7

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)

In [21]:
# ==============================================================================
# 6. TRAINING & EVALUATION LOOP
# ==============================================================================
print("\n--- Training Deep Learning Model (BiLSTM + Attention) ---")
NUM_EPOCHS = 15
best_f1 = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        
    # Eval pada Dev Set
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in dev_loader:
            preds = model(xb.to(DEVICE)).argmax(dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(yb.tolist())
            
    dev_acc = accuracy_score(all_labels, all_preds)
    dev_f1  = f1_score(all_labels, all_preds, average='weighted')
    scheduler.step(dev_f1)
    
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        torch.save(model.state_dict(), 'best_bilstm_fixed.pt')
        marker = '⭐'
    else:
        marker = ''
        
    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Dev Acc: {dev_acc:.4f} | Dev F1w: {dev_f1:.4f} {marker}")


--- Training Deep Learning Model (BiLSTM + Attention) ---
Epoch 01/15 | Loss: 1.4635 | Dev Acc: 0.4215 | Dev F1w: 0.4490 ⭐
Epoch 02/15 | Loss: 1.4667 | Dev Acc: 0.4197 | Dev F1w: 0.4470 
Epoch 03/15 | Loss: 1.4678 | Dev Acc: 0.4206 | Dev F1w: 0.4483 
Epoch 04/15 | Loss: 1.4538 | Dev Acc: 0.4224 | Dev F1w: 0.4495 ⭐
Epoch 05/15 | Loss: 1.4588 | Dev Acc: 0.4215 | Dev F1w: 0.4487 
Epoch 06/15 | Loss: 1.4597 | Dev Acc: 0.4224 | Dev F1w: 0.4491 
Epoch 07/15 | Loss: 1.4639 | Dev Acc: 0.4206 | Dev F1w: 0.4478 
Epoch 08/15 | Loss: 1.4540 | Dev Acc: 0.4215 | Dev F1w: 0.4484 
Epoch 09/15 | Loss: 1.4471 | Dev Acc: 0.4215 | Dev F1w: 0.4482 
Epoch 10/15 | Loss: 1.4525 | Dev Acc: 0.4215 | Dev F1w: 0.4482 
Epoch 11/15 | Loss: 1.4556 | Dev Acc: 0.4224 | Dev F1w: 0.4490 
Epoch 12/15 | Loss: 1.4487 | Dev Acc: 0.4224 | Dev F1w: 0.4490 
Epoch 13/15 | Loss: 1.4593 | Dev Acc: 0.4224 | Dev F1w: 0.4490 
Epoch 14/15 | Loss: 1.4669 | Dev Acc: 0.4224 | Dev F1w: 0.4490 
Epoch 15/15 | Loss: 1.4401 | Dev Acc: 0.422

In [9]:
# ==============================================================================
# 7. FINAL TEST REPORT (HEAD TO HEAD)
# ==============================================================================
print("\n" + "="*50 + "\n FINAL TEST SET PERFORMANCE \n" + "="*50)
model.load_state_dict(torch.load('best_bilstm_fixed.pt'))
model.eval()

lstm_preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        preds = model(xb.to(DEVICE)).argmax(dim=1).cpu()
        lstm_preds.extend(preds.tolist())

print("\n[📊] BI-LSTM + ATTENTION CLASSIFICATION REPORT:")
print(classification_report(y_test, lstm_preds, target_names=le.classes_))

print("\n[📊] NAIVE BAYES CLASSIFICATION REPORT:")
print(classification_report(y_test, nb_preds, target_names=le.classes_))


 FINAL TEST SET PERFORMANCE 

[📊] BI-LSTM + ATTENTION CLASSIFICATION REPORT:
              precision    recall  f1-score   support

       anger       0.40      0.31      0.35       345
     disgust       0.07      0.25      0.11        68
        fear       0.05      0.20      0.07        50
         joy       0.49      0.50      0.50       402
     neutral       0.77      0.54      0.64      1256
     sadness       0.21      0.23      0.22       208
    surprise       0.45      0.58      0.51       281

    accuracy                           0.47      2610
   macro avg       0.35      0.37      0.34      2610
weighted avg       0.57      0.47      0.51      2610


[📊] NAIVE BAYES CLASSIFICATION REPORT:
              precision    recall  f1-score   support

       anger       0.58      0.02      0.04       345
     disgust       0.00      0.00      0.00        68
        fear       0.00      0.00      0.00        50
         joy       0.57      0.11      0.19       402
     neutral  

In [10]:
# ==============================================================================
# 8. FIXED INFERENCE DEMO
# ==============================================================================
print("\n" + "="*50 + "\n 🎯 INFERENCE DEMO (REAL-TIME TEST) \n" + "="*50)

test_sentences = [
    "I am so happy to see you again!",
    "This is absolutely disgusting, I can't believe it.",
    "I'm really scared of what might happen next.",
    "Whatever, I don't really care about it.",
    "How could you do this to me?! I'm furious!",
    "I feel so lonely and sad without you.",
    "Oh my god! I can't believe you're here! What a surprise!",
]

def predict_emotion_fixed(text):
    cleaned = clean_text_for_chat(text)
    tokens = cleaned.split()[:MAX_LEN]
    
    # Map token dengan kamus vocab training
    ids = [vocab.get(t, 1) for t in tokens]
    ids = ids + [0] * (MAX_LEN - len(ids))
    
    x_input = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    
    model.eval()
    with torch.no_grad():
        logits = model(x_input)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
        
    pred_idx = probs.argmax()
    return le.classes_[pred_idx], probs[pred_idx]

for sent in test_sentences:
    emotion, conf = predict_emotion_fixed(sent)
    emoji = {'anger':'😠','disgust':'🤢','fear':'😨','joy':'😊','neutral':'😐','sadness':'😢','surprise':'😲'}.get(emotion,'❓')
    print(f"Teks : {sent}")
    print(f"Hasil: {emoji} {emotion.upper()} ({conf:.2%})\n")


 🎯 INFERENCE DEMO (REAL-TIME TEST) 
Teks : I am so happy to see you again!
Hasil: 😊 JOY (58.91%)

Teks : This is absolutely disgusting, I can't believe it.
Hasil: 😢 SADNESS (54.91%)

Teks : I'm really scared of what might happen next.
Hasil: 😐 NEUTRAL (30.44%)

Teks : Whatever, I don't really care about it.
Hasil: 😢 SADNESS (37.93%)

Teks : How could you do this to me?! I'm furious!
Hasil: 😲 SURPRISE (47.11%)

Teks : I feel so lonely and sad without you.
Hasil: 😢 SADNESS (31.63%)

Teks : Oh my god! I can't believe you're here! What a surprise!
Hasil: 😲 SURPRISE (79.82%)



In [20]:
# ==============================================================================
# 8.3 Interactive — Coba sendiri!
# ==============================================================================
print("\n" + "="*50 + "\n 😎 8.3 INTERACTIVE DEMO \n" + "="*50)

# Ganti teks di bawah ini untuk mencoba kalimat baru:
your_text = "ey yoo look its a-train the fasthest man alive look"

# Map emoji untuk visualisasi hasil akhir
emoji_map = {
    'anger': '😠',
    'disgust': '🤢',
    'fear': '😨',
    'joy': '😊',
    'neutral': '😐',
    'sadness': '😢',
    'surprise': '😲'
}

def predict_emotion_interactive(text):
    cleaned = clean_text_for_chat(text)
    tokens = cleaned.split()[:MAX_LEN]
    
    # Map token dengan kamus vocab training
    ids = [vocab.get(t, 1) for t in tokens]
    ids = ids + [0] * (MAX_LEN - len(ids))
    
    x_input = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    
    model.eval()
    with torch.no_grad():
        logits = model(x_input)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
        
    pred_idx = probs.argmax()
    return le.classes_[pred_idx], probs

# Jalankan Prediksi
predicted_emotion, all_probs = predict_emotion_interactive(your_text)
current_emoji = emoji_map.get(predicted_emotion, '❓')

# Tampilkan Hasil Utama
print(f'Input             : "{your_text}"')
print(f'Predicted Emotion : {current_emoji} {predicted_emotion.upper()}')
print('\nProbabilities Distribution :')

# Tampilkan Bar Chart Horizontal di Terminal
for i, label in enumerate(le.classes_):
    bar_length = int(all_probs[i] * 40)
    bar = '█' * bar_length
    emoji = emoji_map.get(label, '❓')
    print(f'  {emoji} {label:10s} : {all_probs[i]:.4f} {bar}')


 😎 8.3 INTERACTIVE DEMO 
Input             : "ey yoo look its a-train the fasthest man alive look"
Predicted Emotion : 🤢 DISGUST

Probabilities Distribution :
  😠 anger      : 0.0862 ███
  🤢 disgust    : 0.3498 █████████████
  😨 fear       : 0.0032 
  😊 joy        : 0.2299 █████████
  😐 neutral    : 0.1906 ███████
  😢 sadness    : 0.1234 ████
  😲 surprise   : 0.0169 
